# Theme 2: Bug-Fix Quality Over Time

**RQ2:** How does bug-fix *acceptance rate* change over time?  
**RQ3:** How does *time to merge* change over time?  
**RQ4:** How does *patch size* change over time?  
**RQ5:** How does *revision burden* change over time?

Dataset: `mabujadallah/GitHub-Agentic-PR-Dataset`  
Coverage: Dec 2024 – Feb 2026 (15 months)

**Methodology notes:**
- **Confidence bands:** monthly rates carry Wilson 95% CIs; monthly medians carry bootstrap 95% CIs. Bands are shaded around each line.
- **Minimum support:** a monthly cell is reported only if it has ≥ `MIN_N_PROP` (30) PRs for a rate, or ≥ `MIN_N_MEDIAN` (20) for a median. Sparse agent-months (especially Devin) correctly drop out instead of producing 1–2-PR spikes.
- **Repo-matched baseline:** RQ2–RQ4 are re-computed on a repo- and time-matched human set (`build_matched_human_baseline`) so agent-vs-human gaps are not driven by repo mix.
- **Survivorship cutoff** (last 30 days dropped) is applied by `load_fix_prs`.

> **v2 — AIDev repository sample.** This notebook re-runs the analysis restricted to the **true AIDev repository set** (1,743 repos from `hao-li/AIDev`, excl. Codex; 1,615 present in our data) via `load_fix_prs(restrict_to_repos=AIDEV_REPOS_TRUE)`. Figures are written to the `*_figures_v2/` folders. The broad-collection results (v1) are unchanged.

In [1]:
%pip install matplotlib seaborn scipy pyarrow fsspec requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\Mahmoudabujadallah\CLI\.venv\Scripts\python.exe -m pip install --upgrade pip


In [2]:
import sys
sys.path.insert(0, '.')
from analysis_utils import (
    load_fix_prs, load_commits, load_commit_details, build_revision_stats,
    merge_rate, chi_square, mann_whitney, sig_label,
    odds_ratio_ci, cliffs_delta, bh_correct,
    wilson_ci, bootstrap_median_ci, classify_file_role,
    build_matched_human_baseline, annotate_model_releases,
    set_plot_style, save_fig,
    AGENTS, AGENT_COLORS, THEME1_DIR, THEME2_DIR, THEME3_DIR,
    THEME1_DIR_V2, THEME2_DIR_V2, THEME3_DIR_V2, load_aidev_repo_set,
    MIN_N_PROP, MIN_N_MEDIAN,
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
set_plot_style()

In [3]:
AIDEV_REPOS_TRUE = load_aidev_repo_set()
# ── Load data ────────────────────────────────────────────────────────
df         = load_fix_prs(restrict_to_repos=AIDEV_REPOS_TRUE)
agents_df  = df[df['is_agent'] & df['agent'].isin(AGENTS)].copy()
human_df   = df[~df['is_agent']].copy()

# Repo- and time-matched human baseline (for the matched-vs-unmatched comparison)
matched    = build_matched_human_baseline(df)
m_human_df = matched[~matched['is_agent']].copy()

# Commit data needed for RQ4 / RQ5
commits   = load_commits()
details   = load_commit_details()
rev_stats = build_revision_stats(df, commits, details)
print('All data loaded.')

  AIDev repo set loaded: 1,743 repos
Loading fix PRs from HuggingFace ...


  AIDev repo coverage: 8,959 distinct repos
  Survivorship cutoff at 2026-01-29: dropped 33,123 recent PRs
  Restricted to 1,743 given repos: kept 305,739 of 371,577 PRs across 1,585 repos
  Fix PRs loaded: 305,739  |  Agent: 42,284  |  Human: 263,455


  Matched baseline: 42,284 agent PRs + 52,387 matched human PRs (from 263,455 human PRs)
Loading commits from HuggingFace ...


  Commits loaded: 1,156,238
Loading commit details from HuggingFace ...


  Commit details loaded: 7,451,150


All data loaded.


## RQ2: How does bug-fix acceptance rate change over time?

In [4]:
# Monthly merge rate per agent + human baseline, with Wilson 95% CIs.
# A cell is reported only if it has >= MIN_N_PROP PRs.
months = sorted(df['month'].unique())
idx    = [str(m) for m in months]

def _month_grp(m, g, agents_src=agents_df, human_src=human_df):
    if g == 'Human':
        return human_src[human_src['month'] == m]
    return agents_src[(agents_src['month'] == m) & (agents_src['agent'] == g)]

# point estimate + CI bounds per group, kept in parallel dicts for the figure
rate, rate_lo, rate_hi = {}, {}, {}
for g in AGENTS + ['Human']:
    rate[g], rate_lo[g], rate_hi[g] = [], [], []
    for m in months:
        sub = _month_grp(m, g)
        n, k = len(sub), int(sub['is_merged'].sum())
        if n >= MIN_N_PROP:
            lo, hi = wilson_ci(k, n)
            rate[g].append(k / n * 100); rate_lo[g].append(lo); rate_hi[g].append(hi)
        else:
            rate[g].append(None); rate_lo[g].append(None); rate_hi[g].append(None)

monthly_rate = pd.DataFrame(rate, index=idx)
print(f'Monthly merge rate (%), reported only where n >= {MIN_N_PROP}:')
print(monthly_rate.round(1).to_string())

Monthly merge rate (%), reported only where n >= 30:
         Copilot  Cursor  Claude_Code  Devin  Human
2024-12      NaN    94.6         86.2   53.6   87.9
2025-01      NaN    95.9         86.9   34.3   87.3
2025-02      NaN    93.8         85.3   28.0   88.0
2025-03     96.9    94.6         79.1   25.9   87.0
2025-04     94.3    94.4         85.1   38.1   88.0
2025-05     47.4    88.5         85.9   54.3   87.0
2025-06     57.3    86.6         85.6   51.2   85.8
2025-07     56.6    84.9         85.1   52.4   84.4
2025-08     55.5    84.2         86.2   43.8   83.9
2025-09     59.2    89.3         84.8   40.8   84.0
2025-10     65.4    89.3         87.1   49.1   83.5
2025-11     66.5    86.7         84.1   30.5   83.9
2025-12     68.1    89.5         90.5   67.9   83.7
2026-01     68.1    89.2         88.7   72.2   82.8


In [5]:
# Figure: monthly acceptance rate per agent vs human, with Wilson 95% CI bands
x = list(range(len(idx)))
fig, ax = plt.subplots(figsize=(13, 5))
for agent in AGENTS:
    ax.plot(idx, np.array(rate[agent], dtype=float), 'o-',
            color=AGENT_COLORS[agent], label=agent, linewidth=1.8)
    ax.fill_between(x, np.array(rate_lo[agent], dtype=float),
                    np.array(rate_hi[agent], dtype=float),
                    color=AGENT_COLORS[agent], alpha=0.15)
ax.plot(idx, np.array(rate['Human'], dtype=float), 's--',
        color=AGENT_COLORS['Human'], linewidth=2.5, label='Human', zorder=5)
ax.fill_between(x, np.array(rate_lo['Human'], dtype=float),
                np.array(rate_hi['Human'], dtype=float),
                color=AGENT_COLORS['Human'], alpha=0.12)
ax.axvline('2025-07', color='red', linestyle=':', linewidth=1.5, label='AIDev cutoff')
ax.set_xlabel('Month')
ax.set_ylabel('Merge Rate (%)')
ax.set_ylim(0, 105)
ax.set_title('RQ2: Monthly Bug-Fix Acceptance Rate per Agent vs Human (Wilson 95% CI)')
ax.legend()
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
save_fig(fig, 'rq2_monthly_merge_rate', THEME2_DIR_V2)

  -> Saved: results\theme2_figures_v2\rq2_monthly_merge_rate.png


WindowsPath('results/theme2_figures_v2/rq2_monthly_merge_rate.png')

## RQ3: How does time to merge change over time?

In [6]:
# Monthly median time-to-merge per agent + human, with bootstrap 95% CIs.
merged_agents = agents_df[agents_df['is_merged']]
merged_human  = human_df[human_df['is_merged']]

def _month_grp_merged(m, g):
    if g == 'Human':
        return merged_human[merged_human['month'] == m]
    return merged_agents[(merged_agents['month'] == m) & (merged_agents['agent'] == g)]

ttm, ttm_lo, ttm_hi = {}, {}, {}
for g in AGENTS + ['Human']:
    ttm[g], ttm_lo[g], ttm_hi[g] = [], [], []
    for m in months:
        sub = _month_grp_merged(m, g)['hours_to_merge']
        if len(sub) >= MIN_N_MEDIAN:
            lo, hi = bootstrap_median_ci(sub)
            ttm[g].append(round(sub.median(), 2)); ttm_lo[g].append(lo); ttm_hi[g].append(hi)
        else:
            ttm[g].append(None); ttm_lo[g].append(None); ttm_hi[g].append(None)

monthly_ttm = pd.DataFrame(ttm, index=idx)
print(f'Monthly median time to merge (hours), reported only where n >= {MIN_N_MEDIAN}:')
print(monthly_ttm.to_string())

Monthly median time to merge (hours), reported only where n >= 20:
         Copilot  Cursor  Claude_Code  Devin  Human
2024-12      NaN    0.65         3.61   2.41   5.27
2025-01      NaN    0.69         2.84   3.70   5.28
2025-02     0.70    1.02         1.62   0.89   4.44
2025-03     8.95    0.67         2.10   2.45   5.56
2025-04     4.64    0.97         1.83   1.98   4.98
2025-05     8.73    0.93         1.94   0.60   5.41
2025-06    22.67    0.70         0.81   1.82   5.10
2025-07    21.35    0.67         0.84   2.87   5.49
2025-08    17.43    0.77         1.11   1.30   6.09
2025-09    20.18    1.17         1.60   6.53   5.70
2025-10    16.94    1.08         1.79   8.21   6.59
2025-11     8.21    0.78         2.17   0.90   6.22
2025-12     8.08    0.97         1.48   1.19   5.34
2026-01     6.12    0.73         1.35   3.60   5.37


In [7]:
# Figure: monthly time to merge, with bootstrap 95% CI bands
x = list(range(len(idx)))
fig, ax = plt.subplots(figsize=(13, 5))
for agent in AGENTS:
    ax.plot(idx, np.array(ttm[agent], dtype=float), 'o-',
            color=AGENT_COLORS[agent], label=agent, linewidth=1.8)
    ax.fill_between(x, np.array(ttm_lo[agent], dtype=float),
                    np.array(ttm_hi[agent], dtype=float),
                    color=AGENT_COLORS[agent], alpha=0.15)
ax.plot(idx, np.array(ttm['Human'], dtype=float), 's--',
        color=AGENT_COLORS['Human'], linewidth=2.5, label='Human', zorder=5)
ax.fill_between(x, np.array(ttm_lo['Human'], dtype=float),
                np.array(ttm_hi['Human'], dtype=float),
                color=AGENT_COLORS['Human'], alpha=0.12)
ax.axvline('2025-07', color='red', linestyle=':', linewidth=1.5, label='AIDev cutoff')
ax.set_xlabel('Month')
ax.set_ylabel('Median Time to Merge (hours)')
ax.set_title('RQ3: Monthly Median Time to Merge per Agent vs Human (bootstrap 95% CI)')
ax.legend()
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
save_fig(fig, 'rq3_monthly_time_to_merge', THEME2_DIR_V2)

  -> Saved: results\theme2_figures_v2\rq3_monthly_time_to_merge.png


WindowsPath('results/theme2_figures_v2/rq3_monthly_time_to_merge.png')

## RQ4: How does bug-fix patch size change over time?

In [8]:
# Patch size split by file role. A single "lines added" number conflates production
# code, tests, and generated/vendored files; agents and humans write different mixes,
# so we classify each changed file and aggregate lines per role per PR.
details['role'] = details['filename'].map(classify_file_role)

role_size = (
    details.groupby(['pr_id', 'role'])['additions'].sum()
    .unstack(fill_value=0)
    .rename(columns=lambda r: f'{r}_added')
    .reset_index()
    .rename(columns={'pr_id': 'id'})
)
for col in ['prod_added', 'test_added', 'generated_added']:
    if col not in role_size:
        role_size[col] = 0.0
role_size['total_added'] = role_size[['prod_added', 'test_added', 'generated_added']].sum(axis=1)

df_size     = df.merge(role_size, on='id', how='left')
agents_size = df_size[df_size['is_agent'] & df_size['agent'].isin(AGENTS)]
human_size  = df_size[~df_size['is_agent']]

# Monthly median PRODUCTION lines added (the cleaned patch-size signal), bootstrap CIs
psize, psize_lo, psize_hi = {}, {}, {}
for g in AGENTS + ['Human']:
    psize[g], psize_lo[g], psize_hi[g] = [], [], []
    for m in months:
        if g == 'Human':
            sub = human_size[human_size['month'] == m]['prod_added']
        else:
            sub = agents_size[(agents_size['month'] == m) & (agents_size['agent'] == g)]['prod_added']
        if len(sub) >= MIN_N_MEDIAN:
            lo, hi = bootstrap_median_ci(sub)
            psize[g].append(round(sub.median(), 1)); psize_lo[g].append(lo); psize_hi[g].append(hi)
        else:
            psize[g].append(None); psize_lo[g].append(None); psize_hi[g].append(None)

monthly_size = pd.DataFrame(psize, index=idx)
print(f'Monthly median PRODUCTION lines added, reported only where n >= {MIN_N_MEDIAN}:')
print(monthly_size.to_string())

Monthly median PRODUCTION lines added, reported only where n >= 20:
         Copilot  Cursor  Claude_Code  Devin  Human
2024-12      NaN    14.0          7.0   53.0   10.0
2025-01     31.0    10.0          6.0   17.0   11.0
2025-02     18.5    11.0          7.0   41.0   11.0
2025-03     11.5    11.0          5.0   35.0   11.0
2025-04      2.0    13.0          5.0   21.0   11.0
2025-05     19.0    13.0          6.0   30.5   11.0
2025-06     19.0    13.0         10.0   19.0   12.0
2025-07     16.0    16.0         13.0   21.5   12.0
2025-08     18.0    17.0         16.0   15.0   12.0
2025-09     15.0    18.0         15.0   14.0   13.0
2025-10     15.0    18.0         19.0   15.0   12.0
2025-11     10.0    16.0         17.0   28.0   14.0
2025-12     13.5    18.0         17.0   12.0   14.0
2026-01     13.0    21.0         11.0    5.5   14.0


In [9]:
# Figure: monthly PRODUCTION patch size (lines added), bootstrap 95% CI bands
x = list(range(len(idx)))
fig, ax = plt.subplots(figsize=(13, 5))
for agent in AGENTS:
    ax.plot(idx, np.array(psize[agent], dtype=float), 'o-',
            color=AGENT_COLORS[agent], label=agent, linewidth=1.8)
    ax.fill_between(x, np.array(psize_lo[agent], dtype=float),
                    np.array(psize_hi[agent], dtype=float),
                    color=AGENT_COLORS[agent], alpha=0.15)
ax.plot(idx, np.array(psize['Human'], dtype=float), 's--',
        color=AGENT_COLORS['Human'], linewidth=2.5, label='Human', zorder=5)
ax.fill_between(x, np.array(psize_lo['Human'], dtype=float),
                np.array(psize_hi['Human'], dtype=float),
                color=AGENT_COLORS['Human'], alpha=0.12)
ax.axvline('2025-07', color='red', linestyle=':', linewidth=1.5, label='AIDev cutoff')
ax.set_xlabel('Month')
ax.set_ylabel('Median Production Lines Added')
ax.set_title('RQ4: Monthly Median Patch Size — Production Code Only (bootstrap 95% CI)')
ax.legend()
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
save_fig(fig, 'rq4_monthly_patch_size', THEME2_DIR_V2)

  -> Saved: results\theme2_figures_v2\rq4_monthly_patch_size.png


WindowsPath('results/theme2_figures_v2/rq4_monthly_patch_size.png')

In [10]:
# Figure: overall median lines added by file role (prod / test / generated) per group.
# Shows how much of each group's patch is production vs test vs generated code.
roles  = ['prod_added', 'test_added', 'generated_added']
labels = ['Production', 'Test', 'Generated']
groups = AGENTS + ['Human']

medians = {role: [] for role in roles}
for g in groups:
    sub = human_size if g == 'Human' else agents_size[agents_size['agent'] == g]
    for role in roles:
        medians[role].append(sub[role].median())

xpos = np.arange(len(groups))
width = 0.25
fig, ax = plt.subplots(figsize=(11, 5))
for i, (role, lab) in enumerate(zip(roles, labels)):
    ax.bar(xpos + (i - 1) * width, medians[role], width, label=lab)
ax.set_xticks(xpos)
ax.set_xticklabels(groups, rotation=20, ha='right')
ax.set_ylabel('Median Lines Added')
ax.set_title('RQ4: Median Patch Size by File Role per Group')
ax.legend()
fig.tight_layout()
save_fig(fig, 'rq4_patch_size_by_role', THEME2_DIR_V2)

  -> Saved: results\theme2_figures_v2\rq4_patch_size_by_role.png


WindowsPath('results/theme2_figures_v2/rq4_patch_size_by_role.png')

## RQ5: How does revision burden change over time?

In [11]:
# Monthly revision rate: % of merged PRs with >1 commit, per agent, with Wilson 95% CIs.
# NOTE: revision burden is measured from commit count + revision lines only; the dataset
# has no review-comment counts, so review round-trips are not observable (see threats).
agent_rev = rev_stats[rev_stats['agent'].isin(AGENTS)].copy()
agent_rev['is_revised'] = agent_rev['num_commits'] > 1

rev, rev_lo, rev_hi = {}, {}, {}
for g in AGENTS:
    rev[g], rev_lo[g], rev_hi[g] = [], [], []
    for m in months:
        sub = agent_rev[(agent_rev['month'] == m) & (agent_rev['agent'] == g)]
        n, k = len(sub), int(sub['is_revised'].sum())
        if n >= MIN_N_PROP:
            lo, hi = wilson_ci(k, n)
            rev[g].append(round(k / n * 100, 1)); rev_lo[g].append(lo); rev_hi[g].append(hi)
        else:
            rev[g].append(None); rev_lo[g].append(None); rev_hi[g].append(None)

monthly_rev = pd.DataFrame(rev, index=idx)
print(f'Monthly revision rate (%), reported only where n >= {MIN_N_PROP}:')
print(monthly_rev.to_string())

Monthly revision rate (%), reported only where n >= 30:
         Copilot  Cursor  Claude_Code  Devin
2024-12      NaN    44.6         42.2    NaN
2025-01      NaN    42.6         50.9    NaN
2025-02      NaN    43.3         45.7   73.5
2025-03      NaN    46.6         38.6   70.2
2025-04      NaN    49.6         43.1   68.9
2025-05     95.1    45.0         35.2   57.0
2025-06     91.7    43.6         38.8   51.9
2025-07     96.2    52.0         44.6   50.7
2025-08     96.5    51.2         46.2   58.5
2025-09     97.7    53.0         53.7    NaN
2025-10     97.5    52.6         53.1   58.5
2025-11     93.6    48.3         54.3    NaN
2025-12     93.2    49.8         47.0   51.6
2026-01     95.5    49.0         45.7   48.3


In [12]:
# Figure: monthly revision rate, with Wilson 95% CI bands
x = list(range(len(idx)))
fig, ax = plt.subplots(figsize=(13, 5))
for agent in AGENTS:
    ax.plot(idx, np.array(rev[agent], dtype=float), 'o-',
            color=AGENT_COLORS[agent], label=agent, linewidth=1.8)
    ax.fill_between(x, np.array(rev_lo[agent], dtype=float),
                    np.array(rev_hi[agent], dtype=float),
                    color=AGENT_COLORS[agent], alpha=0.15)
ax.axvline('2025-07', color='red', linestyle=':', linewidth=1.5, label='AIDev cutoff')
ax.set_xlabel('Month')
ax.set_ylabel('Revision Rate (%)')
ax.set_title('RQ5: Monthly Revision Rate per Agent (Wilson 95% CI)')
ax.legend()
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
save_fig(fig, 'rq5_monthly_revision_rate', THEME2_DIR_V2)

  -> Saved: results\theme2_figures_v2\rq5_monthly_revision_rate.png


WindowsPath('results/theme2_figures_v2/rq5_monthly_revision_rate.png')

In [13]:
# Monthly median revision lines added (revised PRs only), with bootstrap 95% CIs
reff, reff_lo, reff_hi = {}, {}, {}
for g in AGENTS:
    reff[g], reff_lo[g], reff_hi[g] = [], [], []
    for m in months:
        sub = agent_rev[(agent_rev['month'] == m) & (agent_rev['agent'] == g)
                        & (agent_rev['num_commits'] > 1)]['rev_lines_added']
        if len(sub) >= MIN_N_MEDIAN:
            lo, hi = bootstrap_median_ci(sub)
            reff[g].append(round(sub.median(), 1)); reff_lo[g].append(lo); reff_hi[g].append(hi)
        else:
            reff[g].append(None); reff_lo[g].append(None); reff_hi[g].append(None)

monthly_revsize = pd.DataFrame(reff, index=idx)

x = list(range(len(idx)))
fig, ax = plt.subplots(figsize=(13, 5))
for agent in AGENTS:
    ax.plot(idx, np.array(reff[agent], dtype=float), 'o-',
            color=AGENT_COLORS[agent], label=agent, linewidth=1.8)
    ax.fill_between(x, np.array(reff_lo[agent], dtype=float),
                    np.array(reff_hi[agent], dtype=float),
                    color=AGENT_COLORS[agent], alpha=0.15)
ax.axvline('2025-07', color='red', linestyle=':', linewidth=1.5, label='AIDev cutoff')
ax.set_xlabel('Month')
ax.set_ylabel('Median Revision Lines Added')
ax.set_title('RQ5: Monthly Median Revision Effort — Revised PRs (bootstrap 95% CI)')
ax.legend()
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
save_fig(fig, 'rq5_monthly_revision_effort', THEME2_DIR_V2)

  -> Saved: results\theme2_figures_v2\rq5_monthly_revision_effort.png


WindowsPath('results/theme2_figures_v2/rq5_monthly_revision_effort.png')

## Matched-baseline sensitivity (RQ2)

Agents cluster in particular repos, so an agent-vs-human merge-rate gap can partly reflect *which repos* the comparison spans. Below, each agent's merge rate is tested against both the **unmatched** human baseline (all human PRs) and the **repo+time-matched** baseline. A large shift in the odds ratio between the two columns indicates the unmatched comparison was confounded by repo mix. Odds ratios carry 95% CIs; p-values are Benjamini-Hochberg adjusted across the four agents.

In [14]:
# Agent merge rate vs unmatched and repo+time-matched human baselines.
hu_m, hu_t, hu_r = merge_rate(human_df)     # unmatched humans
hm_m, hm_t, hm_r = merge_rate(m_human_df)   # matched humans

pvals_unm, pvals_mat, rows = [], [], []
for agent in AGENTS:
    sub = agents_df[agents_df['agent'] == agent]
    a_m, a_t, a_r = merge_rate(sub)
    or_u, lo_u, hi_u = odds_ratio_ci(a_m, a_t, hu_m, hu_t)
    or_m, lo_m, hi_m = odds_ratio_ci(a_m, a_t, hm_m, hm_t)
    _, p_u = chi_square(a_m, a_t, hu_m, hu_t); pvals_unm.append(p_u)
    _, p_m = chi_square(a_m, a_t, hm_m, hm_t); pvals_mat.append(p_m)
    rows.append((agent, a_r, or_u, lo_u, hi_u, or_m, lo_m, hi_m))

padj_u = bh_correct(pvals_unm)
padj_m = bh_correct(pvals_mat)

print(f"{'Agent':<13}{'Rate%':>7}   {'OR vs unmatched human':<28}{'OR vs matched human':<28}")
print('-' * 78)
for (agent, a_r, or_u, lo_u, hi_u, or_m, lo_m, hi_m), pu, pm in zip(rows, padj_u, padj_m):
    s_u = f"{or_u:.2f} [{lo_u:.2f},{hi_u:.2f}] {sig_label(pu)}"
    s_m = f"{or_m:.2f} [{lo_m:.2f},{hi_m:.2f}] {sig_label(pm)}"
    print(f"{agent:<13}{a_r:>6.1f}%   {s_u:<28}{s_m:<28}")
print(f"\nHuman merge rate — unmatched: {hu_r:.1f}% (n={hu_t:,})  |  matched: {hm_r:.1f}% (n={hm_t:,})")
print("BH-adjusted p; OR>1 => agent merges more often than that human baseline.")

Agent          Rate%   OR vs unmatched human       OR vs matched human         
------------------------------------------------------------------------------
Copilot        62.3%   0.28 [0.27,0.30] ***        0.26 [0.25,0.28] ***        
Cursor         89.6%   1.47 [1.40,1.55] ***        1.37 [1.30,1.44] ***        
Claude_Code    85.6%   1.02 [0.97,1.07] ns         0.95 [0.90,1.00] *          
Devin          46.7%   0.15 [0.14,0.16] ***        0.14 [0.13,0.15] ***        

Human merge rate — unmatched: 85.4% (n=263,455)  |  matched: 86.3% (n=52,387)
BH-adjusted p; OR>1 => agent merges more often than that human baseline.


## Threats to validity (Theme 2)

- **Merge rate ≠ quality.** A merged PR is not necessarily correct, and a closed-unmerged PR is not necessarily wrong (it may be a duplicate or superseded). RQ2 measures acceptance, not correctness.
- **Auto-merge is unobservable.** The dataset has no `auto_merge` flag, no `merged_by`, and the commits table has no timestamp, so we cannot separate human-reviewed merges from bot/auto-merges. RQ2/RQ3 therefore partly reflect each repo's *merge policy* rather than agent skill — a confound we cannot net out here.
- **Time-to-merge reflects workflow, not just effort.** Fast merges can mean trivial diffs, CI-gated auto-merge, or an attentive maintainer; slow merges can mean a busy maintainer, not a hard fix.
- **Revision burden is commit-based.** `num_commits > 1` is sensitive to force-push and squash-merge policies, and review round-trips (review-comment counts) are absent from the data. Treat RQ5 as a coarse proxy.
- **First-commit ordering is fragile.** `build_revision_stats` takes the "first commit per PR" by parquet row order because the commits table has no commit date; if storage order is not chronological, revision-line attribution can be off.
- **Patch size still imperfect after role split.** `classify_file_role` is heuristic (path patterns); misclassified files leak between prod/test/generated buckets.
- **Repo mix.** Compare the unmatched vs matched odds ratios above; where they diverge, the unmatched agent-vs-human number was partly a repo-composition artefact.